# 🧠 Quant Concepts, Signal Processing & Financial Math Reference

Welcome! This notebook provides interactive, executable code with **artificially generated synthetic data** and charts to visually demonstrate every core quantitative finance concept.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Generate Synthetic Cointegrated Asset Pair (Asset Y and Asset X)
np.random.seed(42)
n_points = 250
X_prices = 50 + np.cumsum(np.random.normal(0, 1, n_points)) # Asset X
beta_true = 1.6 # Hedge Ratio
alpha_true = 12.0
noise = np.random.normal(0, 2.5, n_points) # Epsilon (Spread)
Y_prices = beta_true * X_prices + alpha_true + noise # Asset Y

# 2. Pure NumPy OLS Linear Regression (Zero Statsmodels / Zero SciPy Dependency!)
# Fit Y = Beta * X + Alpha via numpy.polyfit (Degree 1 OLS)
beta, alpha = np.polyfit(X_prices, Y_prices, 1)
spread = Y_prices - (beta * X_prices + alpha) # Vertical distance to line

print(f"--- OLS Quant Glossary Output (Pure NumPy) ---")
print(f"Beta (Hedge Ratio):  {beta:.4f}  -> Trade {beta:.2f} shares of X per 1 share of Y")
print(f"Alpha (Intercept):   {alpha:.4f}  -> Baseline dollar price offset")
print(f"Latest Spread (ε):   ${spread[-1]:.2f} -> Vertical error distance from OLS line")

# 3. Plot OLS Scatter Line & Residual Spread Distance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left Plot: Scatter & OLS Line
ax1.scatter(X_prices, Y_prices, color="blue", alpha=0.5, label="Asset Pair Points")
ax1.plot(X_prices, beta * X_prices + alpha, color="red", linewidth=2, label=f"OLS Line (Y = {beta:.2f}X + {alpha:.1f})")
ax1.set_title("1. OLS Scatter Plot (Asset Y vs Asset X)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Asset X Price ($)")
ax1.set_ylabel("Asset Y Price ($)")
ax1.legend()
ax1.grid(True)

# Right Plot: Vertical Error Distance (Spread Residual ε)
ax2.plot(spread, color="purple", linewidth=1.5, label="Spread Residual ε = Y - (Beta*X + Alpha)")
ax2.axhline(0, color="black", linestyle="--", label="OLS Equilibrium Line (Spread=0)")
ax2.fill_between(range(n_points), spread, 0, where=(spread > 0), color="red", alpha=0.2, label="Y Overpriced (Spread > 0)")
ax2.fill_between(range(n_points), spread, 0, where=(spread < 0), color="green", alpha=0.2, label="Y Underpriced (Spread < 0)")
ax2.set_title("2. Spread Residual Distance (ε) Over Time", fontsize=12, fontweight="bold")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Spread Error ($)")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


## 📌 Section 1: Simple Moving Average (SMA)

### Mathematical Definition:
$$\text{SMA}_k(t) = \frac{1}{k} \sum_{j=0}^{k-1} P_{t-j}$$

An unweighted arithmetic mean of the last $k$ closing price points. It smooths price noise to reveal underlying market direction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
days = 300
dates = pd.date_range(start="2026-01-01", periods=days, freq="B")

returns = np.random.normal(loc=0.0005, scale=0.015, size=days)
price = 100 * np.cumprod(1 + returns)

df_sma = pd.DataFrame({"Close": price}, index=dates)
df_sma['SMA_20'] = df_sma['Close'].rolling(20).mean()
df_sma['SMA_50'] = df_sma['Close'].rolling(50).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_sma.index, df_sma['Close'], label='Synthetic Price', alpha=0.5, color='gray')
plt.plot(df_sma.index, df_sma['SMA_20'], label='SMA 20', color='blue', linewidth=1.5)
plt.plot(df_sma.index, df_sma['SMA_50'], label='SMA 50', color='orange', linewidth=1.5)
plt.title('Section 1: Simple Moving Average (SMA 20 vs SMA 50)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## 📌 Section 2: Exponential Moving Average (EMA)

### Mathematical Definition:
$$\text{EMA}_t = \left(P_t \times \alpha\right) + \left(\text{EMA}_{t-1} \times (1 - \alpha)\right), \quad \alpha = \frac{2}{k+1}$$

Unlike SMA, EMA applies an exponential weight multiplier $\alpha$, giving higher importance to recent price data.

In [ ]:
shock_prices = np.ones(100) * 50
shock_prices[50:] = 80

df_ema = pd.DataFrame({"Close": shock_prices})
df_ema['SMA_20'] = df_ema['Close'].rolling(20).mean()
df_ema['EMA_20'] = df_ema['Close'].ewm(span=20, adjust=False).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_ema['Close'], label='Price (With Artificial Step Shock)', color='black', linewidth=2)
plt.plot(df_ema['SMA_20'], label='20-Day SMA (Slower Response)', color='red', linestyle='--')
plt.plot(df_ema['EMA_20'], label='20-Day EMA (Faster Response)', color='green', linewidth=2)
plt.title('Section 2: SMA vs EMA Lag Comparison on Shock Event', fontsize=13, fontweight='bold')
plt.xlabel('Time Step')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## 📌 Section 3: Digital Signal Processing (DSP) & High-Frequency Noise Filtering

### Advanced Concept: Phase Lag & Low-Pass Filtering
- **Why SMA Lags:** SMA is a rectangular window filter introducing a phase delay of $\frac{k-1}{2}$ days.
- **Savitzky-Golay / Low-Pass Filters:** True DSP filters remove high-frequency noise instantly at the moment of vibration without phase distortion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Generate synthetic price with high-frequency noise & vibration shock
t = np.linspace(0, 10, 200)
clean_signal = np.sin(t) * 10 + 50
vibration_noise = np.random.normal(0, 2, 200)
vibration_noise[100:130] += np.sin(np.arange(30)*1.5) * 8

noisy_price = clean_signal + vibration_noise
df_filter = pd.DataFrame({"Close": noisy_price})
df_filter["SMA_20"] = df_filter["Close"].rolling(20).mean()
df_filter["EMA_20"] = df_filter["Close"].ewm(span=20, adjust=False).mean()

# 2. Zero-SciPy Fallback: Pure NumPy Gaussian Kernel Low-Pass Filter
def numpy_gaussian_lowpass(series, window=21, sigma=3.0):
    x = np.arange(window) - (window - 1) / 2.0
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= kernel.sum()
    return np.convolve(series, kernel, mode="same")

df_filter["LowPass_Gaussian"] = numpy_gaussian_lowpass(df_filter["Close"], window=21, sigma=3.0)

# 3. Plot signal filtering comparison
plt.figure(figsize=(13, 6))
plt.plot(df_filter["Close"], label="Noisy Price Signal (With Vibration)", color="lightgray", linewidth=1.5)
plt.plot(df_filter["SMA_20"], label="SMA 20 (Phase Delay)", color="red", linestyle="--", linewidth=2)
plt.plot(df_filter["EMA_20"], label="EMA 20 (Faster Weighting)", color="orange", linewidth=2)
plt.plot(df_filter["LowPass_Gaussian"], label="Pure NumPy Gaussian Low-Pass (Zero-SciPy Dependency)", color="green", linewidth=2.5)
plt.title("Section 3: Low-Pass Noise Suppression vs SMA Phase Delay (Pure NumPy)", fontsize=13, fontweight="bold")
plt.xlabel("Time Step")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()


--- 
## 📌 Section 3B: Downsampling & Cubic Spline Interpolation

### Concept:
If price data contains high-frequency noise/vibrations, we can **downsample** the series (e.g. taking every 10th sample) to bypass high-frequency noise, then fit a **Cubic Spline Interpolation** (`scipy.interpolate.CubicSpline`) to reconstruct a perfectly smooth continuous trend curve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Downsample: Take a sparse subset (every 10th step)
downsample_step = 10
sparse_indices = df_filter.index[::downsample_step]
sparse_prices = df_filter["Close"].iloc[::downsample_step]

# 2. 100% Pure NumPy Polynomial Fitting (Zero SciPy / Zero Pandas Optional Dependency!)
poly_coeffs = np.polyfit(sparse_indices, sparse_prices, deg=5)
df_filter["Spline_Interpolation"] = np.polyval(poly_coeffs, df_filter.index)

# 3. Plot Downsampled Points vs Pure NumPy Polynomial Trend vs SMA 20
plt.figure(figsize=(13, 6))
plt.plot(df_filter["Close"], label="Original Noisy Price Signal", color="lightgray", linewidth=1.5)
plt.scatter(sparse_indices, sparse_prices, color="red", s=40, zorder=5, label=f"Downsampled Points (Every {downsample_step}th step)")
plt.plot(df_filter["Spline_Interpolation"], label="Pure NumPy Polynomial Interpolation (Zero-SciPy Dependency)", color="blue", linewidth=2.5)
plt.plot(df_filter["SMA_20"], label="SMA 20 (Phase Delay)", color="orange", linestyle="--", linewidth=2)
plt.title("Section 3B: Downsampling & Polynomial Curve Fitting (Pure NumPy)", fontsize=13, fontweight="bold")
plt.xlabel("Time Step")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()


### 📊 Comparison Matrix

| Property | DSP Low-Pass Filtering (Savitzky-Golay / EMA) | Downsampling + Cubic Spline |
| :--- | :--- | :--- |
| **How It Operates** | Processes **every single price tick**, attenuating frequencies above a cutoff. | **Discards intermediate data**, taking sparse sample points (e.g. 1-in-10) and fitting 3rd-degree polynomials. |
| **Real-Time Live Trading** | **High (Causal):** Filters can run strictly on past and present data ($t \le \tau$). | **Low (Non-Causal):** Spline interpolation requires a future boundary point to draw a smooth curve. |
| **Aliasing Risk** | **Low:** Preserves underlying sampling rate safely. | **High:** If downsampling is too aggressive, high-frequency spikes can create fake "phantom" low-frequency waves (**aliasing**). |
| **Curve Smoothness** | Smooth, but tracks localized noise depending on filter window length. | **Extremely Smooth:** Guarantees continuous 1st ($\mathcal{C}^1$) and 2nd ($\mathcal{C}^2$) derivatives. |
| **Primary Quant Use Case** | Real-time high-frequency trading & noise reduction. | Macro-trend modeling, yield curve fitting, and missing data imputation. |

---

### 🔑 Key Quant Takeaway:

1. **Use DSP Low-Pass (Savitzky-Golay / EMA):** When building a **live trading algorithm** that processes data step-by-step in real-time without peeking into the future.
2. **Use Downsampling + Cubic Spline:** When doing **offline quantitative research**, macro trend curve fitting, or constructing smooth yield curves across sparse maturity points.

--- 
## 📌 Section 4: Simple Returns vs. Logarithmic Returns

### 1. Mathematical Definitions
- **Simple Return ($R_{simple}$):** Percentage change over 1 period:
  $$R_{simple} = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$
- **Logarithmic Return ($R_{log}$):** Continuously compounded return:
  $$R_{log} = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

### 2. The Great Flaw of Simple Returns (Non-Symmetry)
Suppose your stock goes from **$\$100 \rightarrow \$50 \rightarrow \$75**:
- **Day 1:** Loss from $\$100$ to $\$50 = -50\%$
- **Day 2:** Gain from $\$50$ to $\$75 = +50\%$
- **Simple Average:** $\frac{-50\% + 50\%}{2} = 0\%$
⚠️ **The Illusion:** Simple average says you broke even ($0\%$), but you started with $\$100$ and ended with $\$75$ (you lost $\$25$)!

### 3. Why Log Returns Fix This (Additive Symmetry)
- **Day 1 Log Return:** $\ln(50 / 100) = -0.6931$
- **Day 2 Log Return:** $\ln(75 / 50) = +0.4055$
- **Total Log Return (Sum):** $-0.6931 + 0.4055 = -0.2877$
✅ **The Truth:** $\exp(-0.2877) \times \$100 = \$75$. Log returns add up perfectly to match real monetary ending value!

In [ ]:
prices = np.array([100.0, 50.0, 75.0])
simple_returns = (prices[1:] - prices[:-1]) / prices[:-1]
log_returns = np.log(prices[1:] / prices[:-1])

print(f"Simple Returns: Day 1 = {simple_returns[0]*100:.1f}%, Day 2 = {simple_returns[1]*100:.1f}%")
print(f"Simple Return Average: {simple_returns.mean()*100:.1f}%")
print(f"Log Returns: Day 1 = {log_returns[0]:.4f}, Day 2 = {log_returns[1]:.4f}")
print(f"Log Return Sum: {log_returns.sum():.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(['Day 1: $100->$50', 'Day 2: $50->$75'], log_returns, color=['red', 'green'], alpha=0.7)
ax.axhline(0, color='black', linestyle=':')
ax.set_title('Section 4: Logarithmic Returns Symmetry', fontsize=13, fontweight='bold')
ax.set_ylabel('Log Return')
plt.grid(True)
plt.show()

--- 
## 📌 Section 5: OLS Linear Regression & Pairs Spread (Complete Glossary & Math)

### 1. The Core Step-by-Step Concept
1. We plot **Asset Y** vs. **Asset X** on a scatter plot.
2. We draw the **OLS Regression Line** ($Y = \beta X + \alpha$).
3. We measure the vertical distance from each point to the OLS line $\implies$ That distance is the **Spread Residual ($\epsilon_t$)**.

### 2. Complete Glossary & Definitions
- **$\beta$ (Hedge Ratio):** The slope of the line. It tells you **how many shares of $X$ to trade for every 1 share of $Y$** so your market risk is zero ($0$ net delta).
  * *Example:* If $\beta = 1.6$, when you Buy 1 share of $Y$, you must Short 1.6 shares of $X$.
- **$\alpha$ (Alpha / Intercept):** The baseline dollar offset between the two assets when $X = 0$.
- **$\epsilon_t$ (Residual / Epsilon / Spread):** The error distance $\epsilon_t = Y_t - (\beta X_t + \alpha)$. It measures pure mispricing.
- **Overpriced ($Y$):** Points lying **above** the OLS line (Spread $> 0$). Asset $Y$ is too expensive relative to $X$.
- **Underpriced ($Y$):** Points lying **below** the OLS line (Spread $< 0$). Asset $Y$ is too cheap relative to $X$.
- **The Trade:** 
  * When $Y$ is overpriced $\rightarrow$ **Short 1 share of $Y$ + Buy $\beta$ shares of $X$**.
  * When $Y$ is underpriced $\rightarrow$ **Buy 1 share of $Y$ + Short $\beta$ shares of $X$**.
- **Hedge**	-	Balancing a risk position so if the overall stock market crashes, your net profit/loss is protected.

### 3. What Does "Trade $\beta$ Shares of X for 1 Share of Y" Mean?

Imagine Pepsi ($Y$) is worth $\$160$ and Coke ($X$) is worth $\$50$, and OLS calculates $\beta = 1.6$.

If the entire stock market crashes by $10\%$:
- Pepsi ($Y$) drops by $1.6 \times$ as much dollar amount as Coke ($X$).
- If you simply bought 1 share of Pepsi and shorted 1 share of Coke, **you would lose money during a market crash!**

#### The Beta Solution:
To be **Market Neutral** (hedged against broad market crashes):
- For every **1 share of Pepsi ($Y$)** you trade...
- You must trade **$1.6$ shares of Coke ($X$)**.

#### How the Trade Works in Practice:

1. **When Pepsi ($Y$) is OVERPRICED ($\epsilon > 2.0$):**
   - **Action:** **SELL 1 Share of Pepsi ($Y$)** AND **BUY $1.6$ Shares of Coke ($X$)**.
   - **Why:** You expect Pepsi to fall back down to the OLS line, and Coke to rise.

2. **When Pepsi ($Y$) is UNDERPRICED ($\epsilon < -2.0$):**
   - **Action:** **BUY 1 Share of Pepsi ($Y$)** AND **SELL / SHORT $1.6$ Shares of Coke ($X$)**.
   - **Why:** You expect Pepsi to rise back up to the OLS line.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Generate Synthetic Cointegrated Asset Pair (Asset Y and Asset X)
np.random.seed(42)
n_points = 250
X_prices = 50 + np.cumsum(np.random.normal(0, 1, n_points)) # Asset X
beta_true = 1.6 # Hedge Ratio
alpha_true = 12.0
noise = np.random.normal(0, 2.5, n_points) # Epsilon (Spread)
Y_prices = beta_true * X_prices + alpha_true + noise # Asset Y

# 2. Pure NumPy OLS Linear Regression (Zero Statsmodels / Zero SciPy Dependency!)
# Fit Y = Beta * X + Alpha via numpy.polyfit (Degree 1 OLS)
beta, alpha = np.polyfit(X_prices, Y_prices, 1)
spread = Y_prices - (beta * X_prices + alpha) # Vertical distance to line

print(f"--- OLS Quant Glossary Output (Pure NumPy) ---")
print(f"Beta (Hedge Ratio):  {beta:.4f}  -> Trade {beta:.2f} shares of X per 1 share of Y")
print(f"Alpha (Intercept):   {alpha:.4f}  -> Baseline dollar price offset")
print(f"Latest Spread (ε):   ${spread[-1]:.2f} -> Vertical error distance from OLS line")

# 3. Plot OLS Scatter Line & Residual Spread Distance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left Plot: Scatter & OLS Line
ax1.scatter(X_prices, Y_prices, color="blue", alpha=0.5, label="Asset Pair Points")
ax1.plot(X_prices, beta * X_prices + alpha, color="red", linewidth=2, label=f"OLS Line (Y = {beta:.2f}X + {alpha:.1f})")
ax1.set_title("1. OLS Scatter Plot (Asset Y vs Asset X)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Asset X Price ($)")
ax1.set_ylabel("Asset Y Price ($)")
ax1.legend()
ax1.grid(True)

# Right Plot: Vertical Error Distance (Spread Residual ε)
ax2.plot(spread, color="purple", linewidth=1.5, label="Spread Residual ε = Y - (Beta*X + Alpha)")
ax2.axhline(0, color="black", linestyle="--", label="OLS Equilibrium Line (Spread=0)")
ax2.fill_between(range(n_points), spread, 0, where=(spread > 0), color="red", alpha=0.2, label="Y Overpriced (Spread > 0)")
ax2.fill_between(range(n_points), spread, 0, where=(spread < 0), color="green", alpha=0.2, label="Y Underpriced (Spread < 0)")
ax2.set_title("2. Spread Residual Distance (ε) Over Time", fontsize=12, fontweight="bold")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Spread Error ($)")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


--- 
## 📌 Section 6: Normalized Z-Score Mean Reversion Signals

### 1. What is a Z-Score?
A **$Z$-score** measures how many standard deviations ($\sigma$) an asset price or spread residual is away from its historical rolling mean ($\mu$).

### 2. Mathematical Formula:
$$Z_t = \frac{\text{Spread}_t - \mu_{\text{spread}}}{\sigma_{\text{spread}}}$$
Where:
- $\text{Spread}_t = Y_t - (\beta X_t + \alpha)$ (The price difference residual between cointegrated assets $Y$ and $X$).
- \mu_{\text{spread}} = Rolling mean of the spread over window $N$.
- \sigma_{\text{spread}} = Rolling standard deviation of the spread over window $N$.

### 3. Quantitative Trading Logic (StatArb Mean Reversion):
- **$Z_t \ge +2.0$ (Overpriced Anomaly):** Asset $Y$ is abnormally expensive relative to $X$. Trigger **Short $Y$ / Long $X$** (Signal = $-1$).
- **$Z_t \le -2.0$ (Underpriced Anomaly):** Asset $Y$ is abnormally cheap relative to $X$. Trigger **Long $Y$ / Short $X$** (Signal = $+1$).
- **$|Z_t| \le 0.5$ (Reverted to Mean):** Spread has snapped back to normal. **Close Position** (Signal = $0$).

In [ ]:
z_score = (spread - np.mean(spread)) / np.std(spread)

plt.figure(figsize=(12, 5))
plt.plot(z_score, color='darkblue', label='Normalized Spread Z-Score')
plt.axhline(2.0, color='red', linestyle='--', label='Short Entry Signal (+2.0 σ)')
plt.axhline(-2.0, color='green', linestyle='--', label='Long Entry Signal (-2.0 σ)')
plt.axhline(0, color='gray', linestyle=':')
plt.title('Section 6: Z-Score Mean-Reversion Signals', fontsize=13, fontweight='bold')
plt.xlabel('Time Step')
plt.ylabel('Z-Score')
plt.legend()
plt.grid(True)
plt.show()

--- 
## 📌 Section 7: Sharpe Ratio & Monte Carlo Stress Testing (Complete Math & Docs)

### 1. What is the Sharpe Ratio?
Absolute profit is meaningless without knowing how much risk was taken to earn it. The **Sharpe Ratio ($S$)** measures **excess return per unit of total risk (volatility)**.

#### Mathematical Formula:
$$S = \frac{\mathbb{E}[R_p - R_f]}{\sigma_p} \times \sqrt{252}$$
Where:
- $R_p$ = Portfolio daily return.
- $R_f$ = Risk-free rate (e.g. US 10-Year Treasury Yield).
- $\sigma_p$ = Daily volatility (standard deviation of daily returns).
- $\sqrt{252}$ = Annualization multiplier (252 trading days in a year).

#### Sharpe Benchmark Ratings:
- $S < 1.0 \implies$ Poor risk-adjusted performance.
- $S \ge 1.0 \implies$ Good / Institutional grade.
- $S \ge 2.0 \implies$ Outstanding (Hedge Fund caliber).

---

### 2. What is Monte Carlo Stress Testing?
Backtesting over a single past historical price path can lead to **overfitting luck**. 

**Monte Carlo simulation** reshuffles historical daily returns $10,000$ times using random sampling (with replacement) to simulate $10,000$ statistically probable parallel universe market paths.

#### Key Output Metrics:
- **95% Value-at-Risk (VaR):** The worst 5% percentile loss expected in a year.
- **Maximum Drawdown (MDD) Distribution:** Peak-to-trough worst-case portfolio drop distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Calculate Sharpe Ratio on Synthetic Strategy Returns
np.random.seed(42)
daily_returns = np.random.normal(loc=0.0008, scale=0.012, size=252)
risk_free_rate = 0.04 / 252 # 4% annual risk-free rate converted to daily

excess_returns = daily_returns - risk_free_rate
sharpe_ratio = (excess_returns.mean() / excess_returns.std()) * np.sqrt(252)

print(f"--- Section 7: Risk Analytics Output ---")
print(f"Mean Daily Return:     {daily_returns.mean()*100:.4f}%")
print(f"Daily Volatility (σ):  {daily_returns.std()*100:.4f}%")
print(f"Annualized Sharpe Ratio: {sharpe_ratio:.4f} (Good Institutional Rating)")

# 2. Run Monte Carlo Simulation (10,000 Resampled Paths)
n_sims = 10000
sample_size = 252 # 1 Trading Year
final_returns = []
max_drawdowns = []

for _ in range(n_sims):
    sim_ret = np.random.choice(daily_returns, size=sample_size, replace=True)
    equity_curve = 10000 * np.cumprod(1 + sim_ret)
    final_returns.append((equity_curve[-1] - 10000) / 10000)
    
    peak = np.maximum.accumulate(equity_curve)
    dd = (equity_curve - peak) / peak
    max_drawdowns.append(dd.min())

final_returns = np.array(final_returns)
max_drawdowns = np.array(max_drawdowns)

# 3. Plot Monte Carlo Histogram Distributions & 95% VaR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left Plot: Return Distribution & 95% VaR
ax1.hist(final_returns * 100, bins=50, color="#1f77b4", edgecolor="black", alpha=0.7)
var_95 = np.percentile(final_returns, 5) * 100
ax1.axvline(var_95, color="red", linestyle="--", linewidth=2, label=f"95% VaR Threshold ({var_95:.1f}%)")
ax1.set_title("1. Monte Carlo: 1-Year Return Distribution (%)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Annual Return (%)")
ax1.set_ylabel("Frequency")
ax1.legend()
ax1.grid(True)

# Right Plot: Max Drawdown Distribution
ax2.hist(max_drawdowns * 100, bins=50, color="#d62728", edgecolor="black", alpha=0.7)
mdd_5 = np.percentile(max_drawdowns, 5) * 100
ax2.axvline(mdd_5, color="black", linestyle="--", linewidth=2, label=f"Worst 5% Drawdown ({mdd_5:.1f}%)")
ax2.set_title("2. Monte Carlo: Max Drawdown Distribution (%)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Max Drawdown (%)")
ax2.set_ylabel("Frequency")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


--- 
## 📌 Section 8: Moving Average Convergence Divergence (MACD)

### 1. What is MACD?
MACD is a trend-following momentum indicator that calculates the relationship between two exponential moving averages (EMA).

### 2. Mathematical Formulas:
- **MACD Line:** $\text{MACD}_t = \text{EMA}_{12}(P_t) - \text{EMA}_{26}(P_t)$
- **Signal Line:** $\text{Signal}_t = \text{EMA}_9(\text{MACD}_t)$
- **MACD Histogram:** $\text{Histogram}_t = \text{MACD}_t - \text{Signal}_t$

### 3. Quantitative Signal Rules:
- **Bullish Crossover:** MACD Line crosses **above** Signal Line $\implies$ Buy Signal ($+1$).
- **Bearish Crossover:** MACD Line crosses **below** Signal Line $\implies$ Sell Signal ($-1$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generate synthetic price data
np.random.seed(42)
prices = 100 * np.cumprod(1 + np.random.normal(0.0005, 0.015, 250))
df_macd = pd.DataFrame({"Close": prices})

# Calculate MACD components
ema12 = df_macd["Close"].ewm(span=12, adjust=False).mean()
ema26 = df_macd["Close"].ewm(span=26, adjust=False).mean()
df_macd["MACD"] = ema12 - ema26
df_macd["Signal_Line"] = df_macd["MACD"].ewm(span=9, adjust=False).mean()
df_macd["Histogram"] = df_macd["MACD"] - df_macd["Signal_Line"]

# Plot MACD
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax1.plot(df_macd["Close"], label="Synthetic Price", color="black")
ax1.set_title("Section 8: MACD Strategy Indicator", fontsize=13, fontweight="bold")
ax1.set_ylabel("Price ($)")
ax1.legend()
ax1.grid(True)

ax2.plot(df_macd["MACD"], label="MACD Line (12-26 EMA)", color="blue", linewidth=1.5)
ax2.plot(df_macd["Signal_Line"], label="Signal Line (9-EMA)", color="red", linestyle="--", linewidth=1.5)
ax2.bar(df_macd.index, df_macd["Histogram"], label="Histogram", color=np.where(df_macd["Histogram"] >= 0, "green", "red"), alpha=0.5)
ax2.axhline(0, color="black", linestyle=":")
ax2.set_ylabel("MACD Value")
ax2.set_xlabel("Time Step")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


--- 
## 📌 Section 9: Bollinger Bands

### 1. What are Bollinger Bands?
Bollinger Bands are volatility bands placed above and below a moving average. Band distance expands during high volatility and contracts during low volatility.

### 2. Mathematical Formulas:
- **Middle Band ($MB$):** $MB_t = \text{SMA}_{20}(P_t)$
- **Upper Band ($UB$):** $UB_t = \text{SMA}_{20}(P_t) + k \times \sigma_{20}(P_t)$
- **Lower Band ($LB$):** $LB_t = \text{SMA}_{20}(P_t) - k \times \sigma_{20}(P_t)$
*(where $k = 2.0$ standard deviations $\sigma$).*

### 3. Quantitative Trading Signals:
- **Touch Lower Band ($P_t \le LB_t$):** Oversold anomaly $\implies$ Buy Signal ($+1$).
- **Touch Upper Band ($P_t \ge UB_t$):** Overbought anomaly $\implies$ Sell Signal ($-1$).

In [ ]:
df_bb = pd.DataFrame({"Close": prices})

# Calculate Bollinger Bands (20-day SMA, 2.0 Std Dev)
df_bb["Middle_Band"] = df_bb["Close"].rolling(20).mean()
rolling_std = df_bb["Close"].rolling(20).std()
df_bb["Upper_Band"] = df_bb["Middle_Band"] + (2.0 * rolling_std)
df_bb["Lower_Band"] = df_bb["Middle_Band"] - (2.0 * rolling_std)

# Plot Bollinger Bands
plt.figure(figsize=(13, 6))
plt.plot(df_bb["Close"], label="Synthetic Price", color="black", linewidth=1.5)
plt.plot(df_bb["Middle_Band"], label="Middle Band (20-SMA)", color="blue", linestyle="--", linewidth=1.5)
plt.plot(df_bb["Upper_Band"], label="Upper Band (+2σ)", color="red", linewidth=1.5)
plt.plot(df_bb["Lower_Band"], label="Lower Band (-2σ)", color="green", linewidth=1.5)
plt.fill_between(df_bb.index, df_bb["Lower_Band"], df_bb["Upper_Band"], color="gray", alpha=0.15, label="Volatility Envelope")
plt.title("Section 9: Bollinger Bands Volatility Envelope", fontsize=13, fontweight="bold")
plt.xlabel("Time Step")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()


--- 
## 📌 Section 10: Relative Strength Index (RSI)

### 1. What is RSI?
RSI is a bounded momentum oscillator (ranging from $0$ to $100$) that measures the speed and change of price movements.

### 2. Mathematical Formulas:
- **Price Difference ($\Delta P$):** $\Delta P_t = P_t - P_{t-1}$
- **Gain ($G$) & Loss ($L$):** $G_t = \max(\Delta P_t, 0)$, $L_t = \max(-\Delta P_t, 0)$
- **Relative Strength ($RS$):** $RS = \frac{\text{EMA}_{14}(G)}{\text{EMA}_{14}(L)}$
- **RSI:** $\text{RSI} = 100 - \left(\frac{100}{1 + RS}\right)$

### 3. Quantitative Trading Signals:
- **RSI $\le 30$ (Oversold):** Price drop is exhausted $\implies$ Buy Signal ($+1$).
- **RSI $\ge 70$ (Overbought):** Price rise is overextended $\implies$ Sell Signal ($-1$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generate synthetic price data
np.random.seed(42)
prices = 100 * np.cumprod(1 + np.random.normal(0.0005, 0.015, 250))
df_rsi = pd.DataFrame({"Close": prices})

# Calculate RSI (14-day window)
delta = df_rsi["Close"].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)

avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()

rs = avg_gain / avg_loss
df_rsi["RSI"] = 100 - (100 / (1 + rs))

# Plot RSI
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax1.plot(df_rsi["Close"], label="Synthetic Price", color="black")
ax1.set_title("Section 10: Relative Strength Index (RSI)", fontsize=13, fontweight="bold")
ax1.set_ylabel("Price ($)")
ax1.legend()
ax1.grid(True)

ax2.plot(df_rsi["RSI"], label="14-Day RSI", color="purple", linewidth=1.5)
ax2.axhline(70, color="red", linestyle="--", label="Overbought (70)")
ax2.axhline(30, color="green", linestyle="--", label="Oversold (30)")
ax2.axhline(50, color="gray", linestyle=":")
ax2.fill_between(df_rsi.index, df_rsi["RSI"], 70, where=(df_rsi["RSI"] >= 70), color="red", alpha=0.3)
ax2.fill_between(df_rsi.index, df_rsi["RSI"], 30, where=(df_rsi["RSI"] <= 30), color="green", alpha=0.3)
ax2.set_ylabel("RSI (0-100)")
ax2.set_xlabel("Time Step")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


--- 
## 📌 Section 11: Trading Volume & Trend Confirmation

### 1. Why Volume Matters in Quants
Volume measures market conviction and liquidity. Price breakouts accompanied by **high volume** confirm true institutional trends, whereas low volume breakouts indicate false traps.

### 2. Volume Metrics & On-Balance Volume (OBV):
$$\text{OBV}_t = \text{OBV}_{t-1} + \begin{cases} +\text{Volume}_t & \text{if } P_t > P_{t-1} \\ -\text{Volume}_t & \text{if } P_t < P_{t-1} \\ 0 & \text{if } P_t = P_{t-1} \end{cases}$$

### 3. Quantitative Rules:
- **Price Up + High Volume Spike:** Strong Bullish Trend Confirmation.
- **Price Down + High Volume Spike:** Strong Bearish Selling Pressure.

In [ ]:
np.random.seed(42)
volume = np.random.randint(100000, 1000000, 250)
df_vol = pd.DataFrame({"Close": prices, "Volume": volume})

# Calculate On-Balance Volume (OBV)
df_vol["Price_Diff"] = df_vol["Close"].diff()
df_vol["OBV_Direction"] = np.where(df_vol["Price_Diff"] > 0, 1, np.where(df_vol["Price_Diff"] < 0, -1, 0))
df_vol["OBV"] = (df_vol["OBV_Direction"] * df_vol["Volume"]).cumsum()

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax1.plot(df_vol["Close"], label="Synthetic Price", color="black")
ax1.set_title("Section 11: Price, Volume & On-Balance Volume (OBV)", fontsize=13, fontweight="bold")
ax1.set_ylabel("Price ($)")
ax1.legend()
ax1.grid(True)

colors = np.where(df_vol["Price_Diff"] >= 0, "green", "red")
ax2.bar(df_vol.index, df_vol["Volume"], color=colors, alpha=0.6, label="Trading Volume")
ax2.set_ylabel("Volume")
ax2.legend()
ax2.grid(True)

ax3.plot(df_vol["OBV"], label="On-Balance Volume (OBV)", color="teal", linewidth=1.5)
ax3.set_ylabel("OBV Cumulative")
ax3.set_xlabel("Time Step")
ax3.legend()
ax3.grid(True)
plt.tight_layout()
plt.show()


--- 
## 📌 Section 12: Support & Resistance Levels

### 1. What are Support & Resistance Levels?
- **Support (Floor):** A price level where buying interest is strong enough to overcome selling pressure, causing the price to bounce back up.
- **Resistance (Ceiling):** A price level where selling pressure overcomes buying interest, causing the price to reject and fall back down.

### 2. Quantitative Detection (Rolling Min/Max Peaks):
- **Rolling Resistance ($R_{window}$):** $R_t = \max_{t-N \le i \le t}(P_i)$
- **Rolling Support ($S_{window}$):** $S_t = \min_{t-N \le i \le t}(P_i)$

### 3. Quantitative Rules:
- **Bounce at Support ($P_t \approx S_t$):** Buy Signal ($+1$).
- **Reject at Resistance ($P_t \approx R_t$):** Sell Signal ($-1$).
- **Breakout ($P_t > R_t$ with High Volume):** Momentum Breakout Signal.

In [ ]:
df_sr = pd.DataFrame({"Close": prices})

# Calculate rolling 30-period Support and Resistance levels
df_sr["Resistance"] = df_sr["Close"].rolling(30).max()
df_sr["Support"] = df_sr["Close"].rolling(30).min()

plt.figure(figsize=(13, 6))
plt.plot(df_sr["Close"], label="Synthetic Price", color="black", linewidth=1.5)
plt.plot(df_sr["Resistance"], label="Rolling Resistance Level (Ceiling)", color="red", linestyle="--", linewidth=1.5)
plt.plot(df_sr["Support"], label="Rolling Support Level (Floor)", color="green", linestyle="--", linewidth=1.5)
plt.fill_between(df_sr.index, df_sr["Support"], df_sr["Resistance"], color="purple", alpha=0.08, label="Trading Channel Range")
plt.title("Section 12: Support & Resistance Channel Levels", fontsize=13, fontweight="bold")
plt.xlabel("Time Step")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()
